In [ ]:
!pip install -r ../requirements.txt

# 03 Multi Step Workflow

대상 workflow
- classify
- route
- search
- answer
- retry

## 왜 classify와 route가 먼저일까?

`classify`는 지금 들어온 질문이 어떤 종류인지 판단하는 단계다.
예를 들어 환불 문의인지, 운영 시간 문의인지, 회사 소개 요청인지 먼저 정리해두면 다음 단계가 단순해진다.

`route`는 분류 결과를 바탕으로 어디로 보낼지 결정하는 단계다.
예를 들어 `refund_policy`로 분류되면 환불 정책 문서를 보고, `contact_info`로 분류되면 연락처 문서를 보게 만들 수 있다.

즉,
- classify = 무엇에 대한 질문인지 정리
- route = 어떤 처리 경로를 탈지 결정

이 두 단계가 잘 되어야 뒤의 `search`, `answer`, `retry`도 안정적으로 동작한다.

In [ ]:
import os
from pathlib import Path
from pprint import pprint

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key) if api_key else None

DATA_DIR = Path("../data")
MODEL_NAME = "gpt-4.1-mini"

docs = {
    "business_hours": (DATA_DIR / "business_hours.txt").read_text(encoding="utf-8").strip(),
    "refund_policy": (DATA_DIR / "refund_policy.txt").read_text(encoding="utf-8").strip(),
    "contact_info": (DATA_DIR / "contact_info.txt").read_text(encoding="utf-8").strip(),
    "company_intro": (DATA_DIR / "company_intro.txt").read_text(encoding="utf-8").strip(),
}

print("OPENAI_API_KEY loaded:", bool(api_key))
print("사용 가능한 문서:", list(docs.keys()))

In [ ]:
for name, text in docs.items():
    preview = text[:120].replace("\n", " ")
    print(f"[{name}]")
    print(preview)
    print()

## 1. Intent 라벨 설계

먼저 질문을 어떤 카테고리로 나눌지 정해야 한다.
라벨이 너무 많으면 분류가 어려워지고, 너무 적으면 서로 다른 질문이 한데 섞인다.

이번 실습에서는 아래 5개 라벨을 사용한다.
- `business_hours`
- `refund_policy`
- `contact_info`
- `company_intro`
- `other`

`other`는 범위 밖 질문이나 애매한 질문을 안전하게 받기 위한 라벨이다.

In [ ]:
LABELS = [
    "business_hours",
    "refund_policy",
    "contact_info",
    "company_intro",
    "other",
]

ROUTE_TABLE = {
    "business_hours": {"type": "document", "target": "business_hours"},
    "refund_policy": {"type": "document", "target": "refund_policy"},
    "contact_info": {"type": "document", "target": "contact_info"},
    "company_intro": {"type": "document", "target": "company_intro"},
    "other": {"type": "fallback", "target": None},
}

pprint(ROUTE_TABLE)

## 2. Rule-based Classify

가장 먼저 해볼 수 있는 방법은 규칙 기반 분류다.
질문 안에 특정 키워드가 있는지 보고 라벨을 붙인다.

장점
- 빠르다
- 동작이 예측 가능하다
- 디버깅이 쉽다

단점
- 표현이 조금만 달라져도 놓칠 수 있다
- 복합 질문 처리에 약하다

In [ ]:
def classify_rule_based(query: str) -> str:
    q = query.lower()

    refund_keywords = ["환불", "취소", "refund"]
    hours_keywords = ["운영", "영업", "몇 시", "시간", "주말"]
    contact_keywords = ["연락처", "전화", "이메일", "메일", "카카오톡", "문의"]
    intro_keywords = ["회사 소개", "무슨 회사", "어떤 회사", "소개", "서비스"]

    if any(keyword in q for keyword in refund_keywords):
        return "refund_policy"
    if any(keyword in q for keyword in hours_keywords):
        return "business_hours"
    if any(keyword in q for keyword in contact_keywords):
        return "contact_info"
    if any(keyword in q for keyword in intro_keywords):
        return "company_intro"
    return "other"

In [ ]:
test_queries = [
    "환불은 언제까지 가능한가요?",
    "고객센터 몇 시까지 운영해요?",
    "문의 이메일 알려주세요.",
    "어떤 회사인지 소개해주세요.",
    "채용 중인가요?",
]

for query in test_queries:
    label = classify_rule_based(query)
    print(f"query: {query}")
    print(f"label: {label}")
    print()

## 3. LLM 기반 Classify

규칙 기반 방식 다음에는 LLM에게 분류를 맡길 수 있다.
이 방식은 표현이 조금 달라져도 더 유연하게 대응할 수 있다.

여기서는 구조화된 출력으로 `label`과 `reason`을 함께 받는다.
이렇게 하면 단순히 결과만 보는 것보다 왜 그렇게 분류했는지도 같이 확인할 수 있다.

`OPENAI_API_KEY`가 설정되어 있을 때만 실행하면 된다.

In [ ]:
from typing import Literal
from pydantic import BaseModel

class ClassificationResult(BaseModel):
    label: Literal[
        "business_hours",
        "refund_policy",
        "contact_info",
        "company_intro",
        "other",
    ]
    reason: str


def classify_with_llm(query: str) -> ClassificationResult:
    if client is None:
        raise ValueError("OPENAI_API_KEY가 없어서 LLM classify를 실행할 수 없습니다.")

    response = client.responses.parse(
        model=MODEL_NAME,
        instructions=(
            "너는 사용자 질문을 intent label로 분류하는 라우팅 전 단계 분류기다. "
            "반드시 주어진 라벨 중 하나만 선택하고, 짧은 이유를 함께 반환하라."
        ),
        input=(
            f"가능한 라벨: {LABELS}\n"
            f"사용자 질문: {query}"
        ),
        text_format=ClassificationResult,
    )
    return response.output_parsed

In [ ]:
# API 키가 있을 때만 실행해보세요.
# result = classify_with_llm("환불 문의는 어디로 하면 되나요?")
# print(result)

## 4. Route

이제 분류 결과를 바탕으로 처리 경로를 정한다.

예를 들어
- `refund_policy`면 환불 정책 문서를 사용
- `contact_info`면 연락처 문서를 사용
- `other`면 fallback 응답으로 보냄

여기서는 route 결과를 딕셔너리로 반환해서, 뒤 단계가 그대로 이어받아 사용할 수 있게 만든다.

In [ ]:
def route_by_label(label: str) -> dict:
    route = ROUTE_TABLE[label]

    if route["type"] == "document":
        target = route["target"]
        return {
            "type": "document",
            "target": target,
            "content": docs[target],
        }

    return {
        "type": "fallback",
        "target": None,
        "content": "현재 준비된 문서 범위를 벗어난 질문입니다.",
    }

In [ ]:
query = "고객센터 운영 시간이 어떻게 되나요?"
label = classify_rule_based(query)
route_result = route_by_label(label)

print("query:", query)
print("label:", label)
print("route type:", route_result["type"])
print("route target:", route_result["target"])
print()
print(route_result["content"][:300])

## 5. classify + route 묶기

실제 멀티스텝 workflow에서는 단계를 따로 만들되, 하나의 함수로도 쉽게 엮을 수 있어야 한다.
아래 함수는 지금 단계에서 `classify -> route`까지만 수행한다.

이후에는 여기에 `search`, `answer`, `retry`를 순서대로 추가하면 된다.

In [ ]:
def run_workflow_preview(query: str, use_llm_classifier: bool = False) -> dict:
    if use_llm_classifier:
        classification = classify_with_llm(query)
        label = classification.label
        reason = classification.reason
    else:
        label = classify_rule_based(query)
        reason = "rule-based classifier"

    route_result = route_by_label(label)

    return {
        "query": query,
        "classification": {
            "label": label,
            "reason": reason,
        },
        "route": route_result,
    }

In [ ]:
sample_queries = [
    "환불은 며칠 안에 요청해야 하나요?",
    "대표 이메일 주소 알려주세요.",
    "주말에도 고객센터 하나요?",
    "당신들은 어떤 서비스를 제공하나요?",
    "채용 공고는 어디서 보나요?",
]

for query in sample_queries:
    result = run_workflow_preview(query)
    print("=" * 60)
    print("query:", result["query"])
    print("label:", result["classification"]["label"])
    print("reason:", result["classification"]["reason"])
    print("route target:", result["route"]["target"])
    print("route type:", result["route"]["type"])

## 6. 다음 단계 확장 포인트

지금은 `classify`와 `route`만 구현했다.
하지만 멀티스텝 workflow 관점에서는 이 뒤에 아래 단계가 자연스럽게 이어진다.

- `search`: route 결과에 따라 벡터 검색, 키워드 검색, DB 조회 등을 실행
- `answer`: 찾은 문서나 도구 결과를 바탕으로 최종 응답 생성
- `retry`: 분류가 애매하거나 검색 결과가 부족할 때 다른 경로로 재시도

즉, 지금 만든 출력 구조는 나중에 이렇게 확장할 수 있다.

`query -> classification -> route -> search -> answer -> retry`

In [ ]:
def search_step(route_result: dict) -> dict:
    return {
        "status": "not_implemented",
        "input_route": route_result,
    }


def answer_step(query: str, search_result: dict) -> dict:
    return {
        "status": "not_implemented",
        "query": query,
        "search_result": search_result,
    }


def retry_step(state: dict) -> dict:
    return {
        "status": "not_implemented",
        "state": state,
    }

## 직접 실험해볼 것

1. `classify_rule_based()`의 키워드를 더 추가해보기
2. `classify_with_llm()` 결과와 rule-based 결과를 비교해보기
3. `환불 문의는 어디로 하나요?`처럼 복합 질문에서 어떤 라벨이 나오는지 확인해보기
4. `other`로 분류된 질문을 어떻게 재시도할지 설계해보기
5. 다음 시간에 `route` 결과를 받아 실제 `search` 단계로 연결해보기

## 오늘 정리

- `classify`는 질문을 구조화하는 단계다.
- `route`는 구조화된 결과를 바탕으로 처리 경로를 고르는 단계다.
- 이 둘을 먼저 분리해두면 이후 `search`, `answer`, `retry`를 붙이기 쉬워진다.
- 멀티스텝 workflow는 각 단계를 독립적으로 설계할수록 디버깅과 확장이 쉬워진다.